In [1]:
import os
import torch
import timm
import torchvision.transforms as T

from wildlife_datasets.datasets import Lynx
from wildlife_tools.data import WildlifeDataset

from wildlife_tools.train.trainer import BasicTrainer
from wildlife_tools.train.objective import ArcFaceLoss
from wildlife_tools.train.callbacks import EpochCheckpoint, EpochLog, EpochCallbacks

from sklearn.preprocessing import LabelEncoder

In [3]:
transform224 = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [4]:
metadata = Lynx("data_rysy/rys_trening_data_Beno")

labels = metadata.df["identity"].values

encoder = LabelEncoder()
metadata.df["label_id"] = encoder.fit_transform(labels)

print("Classes:", len(encoder.classes_))
print("Images:", len(metadata.df))

Classes: 14
Images: 319


In [7]:
train_df = metadata.df.copy()
train_df["identity"] = train_df["label_id"]

train_dataset = WildlifeDataset(
    train_df,
    metadata.root,
    transform=transform224
)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model(
    "hf-hub:BVRA/MegaDescriptor-T-224",
    pretrained=True
)

model = model.to(device)

INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (BVRA/MegaDescriptor-T-224)


In [ ]:
num_classes = len(encoder.classes_)

objective = ArcFaceLoss(
    num_classes=num_classes,
    embedding_size=768
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
callbacks = EpochCallbacks([
    EpochCheckpoint(folder="checkpoints"),
    EpochLog(folder="logs")
])

In [ ]:
trainer = BasicTrainer(
    dataset=train_dataset,
    model=model,
    objective=objective,
    optimizer=optimizer,
    epochs=20,
    device=device,
    batch_size=16,
    num_workers=0,
    epoch_callback=callbacks
)

trainer.train()

Epoch 19: 100%|█████████████████████████████████████████████████████| 20/20 [00:40<00:00,  2.01s/it]


In [ ]:
trainer.save(
    folder="trained_models",
    file_name="megadescriptor_t224_lynx.pth"
)

## Testovanie


In [8]:
import torch
import numpy as np
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.inference import KnnClassifier

# reload clean model
model = timm.create_model(
    "hf-hub:BVRA/MegaDescriptor-T-224",
    pretrained=False
)

# checkpoint = torch.load(
#     "trained_models/megadescriptor_t224_lynx.pth",
#     map_location=device,
#     weights_only=False
# )

# model.load_state_dict(checkpoint["model"])
model = model.to(device)
model.eval()

SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (layers): Sequential(
    (0): SwinTransformerStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU(approximate='none')
            (drop1): 

In [9]:
from proportional_split import proportional_split

df_database, df_query = proportional_split(
    metadata.df,
    query_ratio=0.2,
    seed=42
)

In [10]:
database_dataset = WildlifeDataset(
    df_database,
    metadata.root,
    transform=transform224
)

query_dataset = WildlifeDataset(
    df_query,
    metadata.root,
    transform=transform224
)

In [11]:
extractor = DeepFeatures(
    model,
    batch_size=16,
    device=device,
    num_workers=0,
    pool="max"
)

database_embeddings = extractor(database_dataset)
query_embeddings = extractor(query_dataset)

100%|█████████████████████████████████████████████████████████████████| 5/5 [00:08<00:00,  1.64s/it]


In [12]:
similarity = CosineSimilarity()(query_embeddings, database_embeddings)

classifier = KnnClassifier(
    k=1,
    database_labels=database_dataset.labels_string
)

predictions = classifier(similarity["cosine"])

accuracy = np.mean(
    query_dataset.labels_string == predictions
)

print("Test accuracy:", accuracy)

Test accuracy: 0.4927536231884058


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\wildlife_tools\inference\classifier.py:61: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  results = pd.DataFrame(results).T.fillna(method="ffill").T


In [13]:
print("Overlapping identities:",
      len(set(df_database["identity"])
          & set(df_query["identity"])))

print("Database size:", len(df_database["identity"]))
print("Query size:", len(df_query["identity"]))

print("Unique DB identities:", len(set(df_database["identity"])))
print("Unique Q identities:", len(set(df_query["identity"])))

Overlapping identities: 14
Database size: 250
Query size: 69
Unique DB identities: 14
Unique Q identities: 14
